In [173]:
from sklearn.impute import KNNImputer
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [174]:
# import kagglehub

# # Download latest version
# path = kagglehub.competition_download('spaceship-titanic', output_dir="data")

# print("Path to competition files:", path)

In [175]:
import pandas as pd

test = pd.read_csv('data/test.csv')
train = pd.read_csv('data/train.csv')

test.info()

<class 'pandas.DataFrame'>
RangeIndex: 4277 entries, 0 to 4276
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   4277 non-null   str    
 1   HomePlanet    4190 non-null   str    
 2   CryoSleep     4184 non-null   object 
 3   Cabin         4177 non-null   str    
 4   Destination   4185 non-null   str    
 5   Age           4186 non-null   float64
 6   VIP           4184 non-null   object 
 7   RoomService   4195 non-null   float64
 8   FoodCourt     4171 non-null   float64
 9   ShoppingMall  4179 non-null   float64
 10  Spa           4176 non-null   float64
 11  VRDeck        4197 non-null   float64
 12  Name          4183 non-null   str    
dtypes: float64(6), object(2), str(5)
memory usage: 434.5+ KB


In [176]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB


In [177]:
train.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [178]:
test["Transported"] = False
df = pd.concat([train, test], sort = False)
df.drop(['Name', 'PassengerId'], axis = 1, inplace = True)


In [179]:
df.isna().sum()

HomePlanet      288
CryoSleep       310
Cabin           299
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
dtype: int64

In [180]:
df.head(5)

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [181]:
df[["Deck", "Num", "Side"]] = df["Cabin"].str.split('/', expand = True)
df = df.drop(columns = ["Cabin"])

In [182]:
df.isna().sum()

HomePlanet      288
CryoSleep       310
Destination     274
Age             270
VIP             296
RoomService     263
FoodCourt       289
ShoppingMall    306
Spa             284
VRDeck          268
Transported       0
Deck            299
Num             299
Side            299
dtype: int64

In [183]:
df["Deck"] = df["Deck"].fillna("U")
df["Num"] = df["Num"].fillna(-1)
df["Side"] = df["Side"].fillna("U")

In [184]:
df["Side"].value_counts()

Side
S    6381
P    6290
U     299
Name: count, dtype: int64

In [185]:
df["Deck"] = df["Deck"].map({"F" : 0, "G" : 1, "E" : 2, "B" : 3, "C" : 4, "D" : 5, "A" : 6, "U" : 7, "T" : 8})
df["Side"] = df["Side"].map({"S" : 0, "P" : 1, "U" : -1})

In [186]:
# Q1: If Deck and Side have already been filled, why fill it again with an imputator?
impute_list = ["CryoSleep", "Age", "VIP", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "Deck", "Num", "Side", "VRDeck"]
rest = list(set(df.columns) - set(impute_list))

df_rest = df[rest]
imp = KNNImputer()
df_imputed = imp.fit_transform(df[impute_list])
df_imputed = pd.DataFrame(df_imputed, columns = impute_list)

df = pd.concat([df_rest.reset_index(drop = True), df_imputed.reset_index(drop = True)], axis = 1)

In [187]:
df["HomePlanet"] = df["HomePlanet"].fillna("U")
df["Destination"] = df["Destination"].fillna("U")
category_cols = ["HomePlanet", "Destination"]

for col in category_cols:
    df = pd.concat([df, pd.get_dummies(df[col], prefix = col, dtype=int)], axis = 1)

In [188]:
df = df.drop(columns = category_cols)

In [189]:
df.head()

,Transported,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,Deck,Num,Side,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,HomePlanet_U,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Destination_U
0,False,0.0,39.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,1.0,0.0,0,1,0,0,0,0,1,0
1,True,0.0,24.0,0.0,109.0,9.0,25.0,549.0,0.0,0.0,0.0,44.0,1,0,0,0,0,0,1,0
2,False,0.0,58.0,1.0,43.0,3576.0,0.0,6715.0,6.0,0.0,0.0,49.0,0,1,0,0,0,0,1,0
3,False,0.0,33.0,0.0,0.0,1283.0,371.0,3329.0,6.0,0.0,0.0,193.0,0,1,0,0,0,0,1,0
4,True,0.0,16.0,0.0,303.0,70.0,151.0,565.0,0.0,1.0,0.0,2.0,1,0,0,0,0,0,1,0


In [190]:
df.isna().sum()

Transported                  0
CryoSleep                    0
Age                          0
VIP                          0
RoomService                  0
FoodCourt                    0
ShoppingMall                 0
Spa                          0
Deck                         0
Num                          0
Side                         0
VRDeck                       0
HomePlanet_Earth             0
HomePlanet_Europa            0
HomePlanet_Mars              0
HomePlanet_U                 0
Destination_55 Cancri e      0
Destination_PSO J318.5-22    0
Destination_TRAPPIST-1e      0
Destination_U                0
dtype: int64

In [195]:
# feature engineering?
bill_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
df["Amt_spend"] = df[bill_cols].sum(axis = 1)
df["Std_amt_spend"] = df[bill_cols].std(axis = 1)
df["Mean_amt_spend"] = df[bill_cols].mean(axis = 1)

df['3_high_cols'] = df['CryoSleep'] + df['HomePlanet_Europa'] + df['Destination_55 Cancri e']
df['3_low_cols'] = df["Mean_amt_spend"] + df["Amt_spend"] + df['HomePlanet_Earth']

In [199]:
df.corr()["Transported"].sort_values(ascending=False)

Transported                  1.000000
CryoSleep                    0.324480
3_high_cols                  0.284212
HomePlanet_Europa            0.131977
Destination_55 Cancri e      0.083625
Deck                         0.062790
FoodCourt                    0.034771
HomePlanet_U                 0.006403
HomePlanet_Mars              0.005643
ShoppingMall                 0.004163
Destination_PSO J318.5-22    0.000760
Destination_U               -0.000554
VIP                         -0.018644
Num                         -0.035240
Age                         -0.050613
Side                        -0.067358
Destination_TRAPPIST-1e     -0.072731
HomePlanet_Earth            -0.119644
Std_amt_spend               -0.121149
Amt_spend                   -0.140423
Mean_amt_spend              -0.140423
3_low_cols                  -0.140447
VRDeck                      -0.142783
Spa                         -0.154825
RoomService                 -0.174742
Name: Transported, dtype: float64

In [202]:
df_train, df_test = df[:train.shape[0]], df[train.shape[0]:]
df_test = df_test.drop(columns = ["Transported"])
df_train.shape, df_test.shape

((8693, 25), (4277, 24))

In [211]:
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [213]:
X = df_train.drop(columns = ["Transported"])
y = df_train["Transported"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

model_1 = LogisticRegression()
model_2 = DecisionTreeClassifier()
model_3 = RandomForestClassifier()
model_4 = XGBClassifier()
model_5 = LGBMClassifier()

In [216]:
model_1.fit(X_train, y_train)
pred = model_1.predict(X_test)
accuracy_score(y_test, pred)

/Users/chinmunyau/Coding Stuff/[04] AI/05 - ICAIC/JupyterNotebooks-MY/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7722829212190915

In [219]:
model_2.fit(X_train, y_train)
pred = model_2.predict(X_test)
accuracy_score(y_test, pred)

0.7343300747556066

In [221]:
model_3.fit(X_train, y_train)
pred = model_3.predict(X_test)
accuracy_score(y_test, pred)

0.7860839562967222

In [222]:
model_4.fit(X_train, y_train)
pred = model_4.predict(X_test)
accuracy_score(y_test, pred)

0.7952846463484762

In [230]:
model_5.fit(X_train, y_train)
pred = model_5.predict(X_test)
accuracy_score(y_test, pred)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 3500, number of negative: 3454
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001652 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2703
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503307 -> initscore=0.013230
[LightGBM] [Info] Start training from score 0.013230


0.7993099482461185

In [226]:
df_dummy = pd.read_csv('data/test.csv')
pred = model_5.predict(df_test)

final = pd.DataFrame()
final['PassengerId'] = df_dummy['PassengerId']
final['Transported'] = pred

final.to_csv('submission.csv', index = False)